## <a href="https://cursos.alura.com.br/course/langchain-desenvolva-agentes-inteligencia-artificial/task/161400"><b>Langchain Agentes - Implementando um agente para extrair dados de universidades</b></a><br/>

<b>Objetivos:</b><br/>
<ol>
    <li>Criação de ferramenta para extração de informação de Universidades, <b><u>mesmo que o nome delas estejam incorreto.</u></b></li>
    <li>Uso do da ferramenta TavilySearch para busca Web, caso a universidade não exista no CSV.</li>    
</ol>
<ul><ul>
    <li>Por isso o uso de LLM nessa ferramenta, além da LLM na ferramenta PerfilAcadêmico.</li>
</ul></ul>
<b>O TavilySearch não é gratuito, possui limitação de créditos para uso.</b>

In [1]:
#%pip install -r requirements.txt

In [2]:
from typing import List
from pydantic import BaseModel, Field

class Nota(BaseModel):
    area_de_conhecimento: str = Field(description="Área de conhecimento da nota")
    nota: float = Field(description="Nota obtida pelo estudante nessa área de conhecimento")
    
class ExtratorPerfilAcademicoDeEstudante(BaseModel):
    
    nome:str = Field(description="Nome do estudante")
    ano_de_conclusao: int = Field(description="Ano de formatura")
    notas: List[Nota] = Field(description="Lista de notas para cada área de conhecimento") # O Nota dentro da lista, é a classe criada acima, que 
                                                                                           # formata o dicionário de notas
                                                                                           
    resumo: str = Field(description="Resumo das principais características desse estudante de forma a torná-lo único e um ótimo potencial estudante para faculdades. Só esse estudante tem bla bla bla")

### <b>CRIAÇÃO DE FERRAMENTAS</b>

<b>1) Criação da Ferramenta DadosDeEstudante</b>

In [3]:
from langchain.tools import BaseTool
from pandas import read_csv
import json

# FERRAMENTA DADOS DE ESTUDANTE
class DadosDeEstudante(BaseTool): # ESTENDE BaseTool
    
    # TODA FERRAMENTA PRECISA TER ESSES ATRIBUTOS
    name: str = "dados_de_estudante" # Nome da ferramenta
    description : str = """ 
                            - Essa ferramenta extrai o histórico e preferências de um estudante, de acordo com o seu histórico.
                            - Passe para essa ferramenta como argumento o nome do estudante. 
                        """ # Descrição da ferramenta
                            # Melhorar o Prompt para se evitar alucinações, dados inventados ou erros de retorno.
    
    def __init__(self): 
        super().__init__() # PARA NÃO SOBRESCREVER O CONSTRUTOR DA CLASSE MÃE BaseTool
                
        print('Inicializando ferramenta Dados de Estudante')      
    
    def __busca_dados_de_estudante(self,estudante:str) -> str:
        
        dfestudantes = read_csv("documentos/estudantes.csv")
        
        dados_estudante = dfestudantes.loc[dfestudantes['USUARIO'] == estudante]
        
        if dados_estudante.empty:
            return f"Desculpe, não encontrei dados para o estudante '{estudante}'. Por favor, verifique o nome e tente novamente.\n"
        
        json_estudante = json.dumps(dados_estudante.to_dict(orient='records')[0],ensure_ascii=False) # MELHOR FORMATO PARA RETORNAR OS DADOS ENTRE FERRAMENTAS. 
                                                                                                     # Ensure ASCII FALSE PARA ACEITAR CARACTERES ESPECIAIS
        
        return json_estudante
        
                
    # CONTRATO DA FERRAMENTA - O QUE ELA FAZ
    def _run(self, input: str) -> str:     
        
        # MODIFICAÇÃO
        estudante = input
                
        print(f'\nRetorno estudante {estudante} da LLM')        
        
        dados = self.__busca_dados_de_estudante(estudante.lower())
        print('\nJSON retornado pelo método busca_dados_de_estudante:', dados)
               
        return dados


<b>2) Criação da Ferramenta Perfil Acadêmico</b>

A utilização da LLM é porque é necessária a criação de um perfil acadêmico com sugestão dela.

In [4]:
from langchain.tools import BaseTool
from langchain.prompts import PromptTemplate
from langchain_core.output_parsers import JsonOutputParser
from langchain_openai import ChatOpenAI
    
class PerfilAcademico(BaseTool): # ESTENDE BaseTool
    
    llm:ChatOpenAI = None
    
    # TODA FERRAMENTA PRECISA TER ESSES ATRIBUTOS
    name: str = "perfil_academico" # Nome da ferramenta
    description : str = """                                                       
                            - Esta ferramenta utiliza os dados do estudante para gerar um perfil acadêmico detalhado.  
                            - Não consigo obter os dados do estudante sozinho. **NUNCA** utilizar somente o nome, execute a ferramenta **dados_de_estudante** 
                            antes para obter os dados.                                         
                        """ # Descrição da ferramenta    
    
    def __init__(self,llm:ChatOpenAI):
        super().__init__() # PARA NÃO SOBRESCREVER O CONSTRUTOR DA CLASSE MÃE BaseTool
        self.llm = llm   
        print('Inicializando ferramenta Perfil Acadêmico')
        
                
    # CONTRATO DA FERRAMENTA - O QUE ELA FAZ
    #
    # JÁ QUE SE TRATA DA ENTRADA DOS DADOS DE UM ESTUDANTE NO FORMATO TEXTO, SERÁ UTILIZADA A SAÍDA DA FERRAMENTA DE DADOS DE ESTUDANTE 
    # COMO ENTRADA DESTA FERRAMENTA DE PERFIL ACADÊMICO.
    #
    # ASSIM, A ENTRADA DESTE MÉTODO _run SERÁ O TEXTO COM OS DADOS DO ESTUDANTE, COMO CONTEXTO DE UM PROMPT, QUE SERÁ USADO 
    # PARA INFORMAR A MANEIRA COMO O PERFIL ACADÊMICO DEVE SER GERADO.
    # 
    def _run(self, input: str) -> str:        
        
        parseador = JsonOutputParser(pydantic_object=ExtratorPerfilAcademicoDeEstudante)    
        
        template = PromptTemplate(
                                    template = """ 
                                                    CONTEXTO:                            
                                                    Você é uma consultora de carreiras
                                                                                
                                                    Esta ferramenta utiliza os dados do estudante para gerar um perfil acadêmico detalhado.
                                                    
                                                    OBJETIVO:
                                                        - Criar o perfil acadêmico de um estudante utilizando os dados fornecidos 
                                                        
                                                        ENTRADA:
                                                        -------------------------
                                                        {dados_do_estudante}
                                                        -------------------------
                                                    
                                                    ESTILO:
                                                    Precisa indicar com detalhes, riqueza, mas direta ao ponto.
                                                                                
                                                    PASSOS:
                                                        - Formate o estudante para o seu perfil acadêmico.
                                                        - Com os dados, identifique as opções de universidades sugeridas e cursos compatíveis com o interesse do aluno.
                                                        - Destaque o perfil do aluno, dando ênfase, principalmente, naquilo que faz interesse nas instituições de interesse
                                                        do aluno.                                                      
                                            
                                                    FORMATO DE SAIDA:
                                                    {formato_saida} 
                                                    
                                                    - Saída em português.
                                                """,
                                    input_variables = ["dados_do_estudante"],
                                    partial_variables = {"formato_saida": parseador.get_format_instructions()}
                                 )
        
        cadeia = template | self.llm | parseador        
        
        print("\nEntrada Perfil Acadêmico\n", input)
        
        resposta = cadeia.invoke({"dados_do_estudante": input})
        
        print('Resposta Perfil Acadêmico\n',resposta)
        
        return resposta


<b>3) Criação da Ferramenta DadosDaUniversidade</b>

<ul><li>Criação de ferramenta para extração de informação de Universidades, <b><u>mesmo que o nome delas estejam incorreto.</u></b></li></ul>

<ul><ul>
    <li>Por isso o uso de LLM nessa ferramenta</li>
</ul></ul>

In [5]:
from langchain.tools import BaseTool
from pandas import read_csv
import json
from langchain.prompts import PromptTemplate
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import JsonOutputParser
from pydantic import BaseModel, Field


class ExtratorNomeDaUniversidade(BaseModel):
    nome_universidade: str = Field(description="Nome da universidade")

# FERRAMENTA DADOS DA UNIVERSIDADE
class DadosDaUniversidade(BaseTool): # ESTENDE BaseTool
    
    llm:ChatOpenAI = None
    
    # TODA FERRAMENTA PRECISA TER ESSES ATRIBUTOS
    name: str = "dados_da_universidade" # Nome da ferramenta
    description : str = """ 
                            - Essa ferramenta extrai os dados de uma universidade.
                            - Passe para essa ferramenta como argumento o nome da universidade.
                            - Caso os dados da universidade não sejam encontrados, executar a ferramenta **dados_da_universidade_web**, passando como argumento o nome da universidade. 
                        """ # Descrição da ferramenta                            
    
    def __init__(self,llm:ChatOpenAI): 
        super().__init__() # PARA NÃO SOBRESCREVER O CONSTRUTOR DA CLASSE MÃE BaseTool
        
        self.llm = llm
        print('Inicializando ferramenta Dados Da Universidade')      
    
    def __busca_dados_da_universidade(self,universidade:str) -> str:
        
        dfuniversidades = read_csv("documentos/universidades.csv")
        
        dfuniversidades['NOME_FACULDADE'] = dfuniversidades['NOME_FACULDADE'].str.lower()
        
        dados_universidade = dfuniversidades.loc[dfuniversidades['NOME_FACULDADE'] == universidade]
        
        if dados_universidade.empty:
            return f"Desculpe, não encontrei dados para a universidades '{universidade}'. Por favor, verifique o nome e tente novamente.\n"
        
        json_universidade = json.dumps(dados_universidade.to_dict(orient='records')[0], ensure_ascii=False) # MELHOR FORMATO PARA RETORNAR OS DADOS ENTRE FERRAMENTAS
        
        return json_universidade
    
    
    def __busca_todas_universidades(self) -> str:
        
        dfuniversidades = read_csv("documentos/universidades.csv")
        
        lista_universidades = []
        
        lista_dict = dfuniversidades.to_dict(orient='records')

        #print('\nLista de dicionário de todas as universidades:\n', lista_dict)
        
        for u in lista_dict:
            lista_universidades.append(json.dumps(u, ensure_ascii=False)) # Ensure ASCII FALSE PARA ACEITAR CARACTERES ESPECIAIS
        
        #print('\nLista JSONs de todas as universidades:\n', lista_universidades)
        
        return lista_universidades
            
                
    # CONTRATO DA FERRAMENTA - O QUE ELA FAZ
    def _run(self, input: str) -> str:     
        
        # MODIFICAÇÃO
        universidade = input
        
        parseador = JsonOutputParser(pydantic_object=ExtratorNomeDaUniversidade)
                
        template = PromptTemplate(
                                    template = """ 
                                                    CONTEXTO:
                                                    ---------------------------                                                    
                                                        {todas_universidades}
                                                    ---------------------------                            
                                                                                                                                    
                                                    ENTRADA:
                                                    ---------------------------
                                                        {nome_da_universidade}
                                                    ---------------------------
                                                    
                                                    OBJETIVO:
                                                        - Fornecer o nome **CORRETO** da universidade com base no contexto fornecido.                                                    
                                                                                                           
                                                    - SAÍDA:
                                                    ----------------------------
                                                        {formato_saida}
                                                    ----------------------------
                                                """,
                                    input_variables = ["nome_da_universidade","todas_universidades"],
                                    partial_variables= {"formato_saida": parseador.get_format_instructions()}
                                 )
        
        cadeia = template | self.llm | parseador
        resposta = cadeia.invoke({"nome_da_universidade": universidade, "todas_universidades": self.__busca_todas_universidades()})['nome_universidade']
        
        print(f'\nRetorno nome da universidade {universidade} da LLM: ', resposta)        
        
        dados = self.__busca_dados_da_universidade(resposta.lower())
        print('\nJSON retornado pelo método busca_dados_da_universidade:', dados)
               
        return dados


In [6]:
from langchain_community.tools.tavily_search import TavilySearchResults
from langchain_openai import ChatOpenAI
from langchain.tools import BaseTool
from langchain.prompts import PromptTemplate
from pydantic import BaseModel, Field
import json
from langchain_core.output_parsers import JsonOutputParser
from typing import Dict

class UniversidadeWeb(BaseModel):
    NOME_UNIVERSIDADE: str = Field(description="Nome da faculdade")
    PAIS: str = Field(description="País onde a faculdade está localizada")
    CRITERIOS_SELECAO: str = Field(description="Critérios de seleção para admissão na faculdade")
    CURSOS_DESTAQUE: str = Field(description="Cursos de destaque oferecidos pela faculdade")
    PERFIL_DESEJADO: str = Field(description="Perfil desejado dos estudantes para admissão na faculdade")

class ExtratorDadosDaUniversidadeWeb(BaseModel):
    dict_universidade_web: Dict[UniversidadeWeb,str] = Field(description="Dicionário com os dados da universidade extraídos da web")


# FERRAMENTA DADOS DA UNIVERSIDADE WEB
class DadosDaUniversidadeWeb(BaseTool): # ESTENDE BaseTool
    
    llm:ChatOpenAI = None
    search:TavilySearchResults = None
    
    # TODA FERRAMENTA PRECISA TER ESSES ATRIBUTOS
    name: str = "dados_da_universidade_web" # Nome da ferramenta
    description : str = """ 
                            - Essa ferramenta extrai os dados de uma universidade da internet.
                            - Passe para essa ferramenta como argumento o nome da universidade. 
                        """ # Descrição da ferramenta                            
    
    def __init__(self,llm:ChatOpenAI): 
        super().__init__() # PARA NÃO SOBRESCREVER O CONSTRUTOR DA CLASSE MÃE BaseTool
        
        self.search = TavilySearchResults()
        
        self.llm = llm
        print('Inicializando ferramenta Dados Da Universidade Web')
                  
                
    # CONTRATO DA FERRAMENTA - O QUE ELA FAZ
    def _run(self, input: str) -> str:     
        
        # MODIFICAÇÃO
        universidade = input
        
        parseador = JsonOutputParser(pydantic_object=ExtratorDadosDaUniversidadeWeb)
        
        template = PromptTemplate(
                                    template = """ 
                                                    ENTRADAS:
                                                    ------------------------------------
                                                        - {nome_da_universidade}
                                                        - {dados_da_universidade_web}
                                                    ------------------------------------
                                                    
                                                    OBJETIVO:
                                                        - Fornecer os dados da universidade considerando o seu nome correto e as ENTRADAS.                                                    
                                                                                                           
                                                    - SAÍDA:
                                                    ----------------------------
                                                        {formato_saida}
                                                    ----------------------------
                                                """,
                                    input_variables = ["nome_da_universidade","todas_universidades"],
                                    partial_variables= {"formato_saida": parseador.get_format_instructions()}
                                 )
        
        cadeia = template | self.llm | parseador
        
        resposta_web = self.search.invoke(f"Obtenha os dados da universidade {universidade} para preencher os campos a seguir para um dicionário. Campos: NOME_FACULDADE, PAIS, CRITERIOS_SELECAO, Notas do Ensino Médio, CURSOS_DESTAQUE, PERFIL_DESEJADO")
        resposta = json.dumps(cadeia.invoke({"nome_da_universidade": universidade, "dados_da_universidade_web": resposta_web})['dict_universidade_web'], ensure_ascii=False) # ENSURE ASCII FALSE PARA ACEITAR CARACTERES ESPECIAIS
        
        
        print('\nJSON retornado: ', resposta)
               
        return resposta

<b>4) Instanciando as Ferramentas que a LLM precisa usar</b>

In [7]:
from langchain.agents import Tool

class Tools:
        
        def __init__(self,llm:ChatOpenAI):
                
                dados_de_estudante = DadosDeEstudante()
                perfil_academico = PerfilAcademico(llm) # INSTANCIANDO O OBJETO DA MINHA FERRAMENTA
                dados_da_universidade = DadosDaUniversidade(llm)
                dados_da_universidade_web = DadosDaUniversidadeWeb(llm)

                # MATRIZ DE FERRAMENTAS (CONJUNTO DE FERRAMENTAS)
                self.tools = [
                        # Instanciando ferramentas
                        Tool(
                                name=dados_de_estudante.name,
                                func=dados_de_estudante.run,
                                description=dados_de_estudante.description,
                                return_direct=False # SE FALSE -> O AGENTE PASSA PELA FASE DE RACIOCÍNIO ANTES DE RETORNAR A RESPOSTA. USADO NA FERRAMENTA INTERMEDIÁRIA
                                                    # SE TRUE -> O AGENTE RETORNA DIRETAMENTE A RESPOSTA DA FERRAMENTA PARA O USUÁRIO, SEM PASSAR PELO RACIÓCINIO DO AGENTE.                                                                                            
                        ),
                        
                        # SEGUNDA FERRAMENTA. 
                        # SE VIRA PARA PEGAR OS DADOS DO ESTUDANTE E GERAR O PERFIL ACADÊMICO
                        Tool(
                                name=perfil_academico.name,
                                func=perfil_academico.run,
                                description=perfil_academico.description                                                      
                        ),
                        
                        # TERCEIRA FERRAMENTA. 
                        # SE VIRA PARA PEGAR OS DADOS DA UNIVERSIDADE E FORNECER SUAS INFORMAÇÕES
                        Tool(
                                name=dados_da_universidade.name,
                                func=dados_da_universidade.run,
                                description=dados_da_universidade.description                                                      
                        ),
                        
                        # QUARTA FERRAMENTA. 
                        # SE VIRA PARA PEGAR OS DADOS DA UNIVERSIDADE E FORNECER SUAS INFORMAÇÕES
                        Tool(
                                name=dados_da_universidade_web.name,
                                func=dados_da_universidade_web.run,
                                description=dados_da_universidade_web.description                                                                                
                        )
                ]

### <b>CRIAÇÃO DO AGENTE DE FERRAMENTAS</b>

<b>5) Informando para a LLM as ferramentas que eu tenho</b> 
<ul><li>Para isso, é necessário criar um agente com as ferramentas</li></ul>

In [8]:
#from langchain.agents import create_openai_tools_agent
from langchain.agents import create_react_agent
from langchain import hub
from dotenv import load_dotenv
from os import getenv
import warnings

warnings.filterwarnings("ignore")

class AgenteReAct:
    
    def __init__(self):
      
      load_dotenv()

      llm = ChatOpenAI(
                        model="gpt-4.1-mini", # TIVE QUE TROCAR PARA UM MODELO MENOR, POR CAUSA DO ERRO ABAIXO
                                              #
                                              #  BadRequestError: Error code: 400 - {'error': {'message': "Unsupported parameter: 'stop' is not supported with this model.", 
                                              #  'type': 'invalid_request_error', 'param': 'stop', 'code': 'unsupported_parameter'}}
                                              #
                        api_key=getenv("API_KEY") 
                      )
      
      # INSTANCIANDO AS FERRAMENTAS
      self.tools = Tools(llm).tools
        
      # PROMPT DE INICIALIZAÇÃO PARA INFORMAR PARA A LLM SOBRE A FERRAMENTA.
      # openai-functions-agent É UM PROMPT PRONTO PARA AGENTES QUE UTILIZAM FUNÇÕES (TOOLS)
      #prompt=(hub.pull(owner_repo_commit="hwchase17/openai-functions-agent"))
      
      # react-agent É UM PROMPT PRONTO PARA AGENTES QUE UTILIZAM O PARADIGMA ReAct (Raciocínio e Ação)
      prompt=(hub.pull(owner_repo_commit="hwchase17/react"))
      

      # CRIANDO UM AGENTE COM AS FERRAMENTAS
      self.agente = create_react_agent(
                                        llm=llm, # INFORMA A LLM QUE VAI SER USADA PELO AGENTE
                                        tools=self.tools, # PASSANDO PARA A LLM A FERRAMENTA QUE ELA PODE USAR. INSTÂNCIA DA FERRAMENTA
                                        prompt=prompt  
                                                          # JÁ EXISTEM PROMPTS PRONTOS NO REPOSITÓRIO DO LANGSMITH, DE ACORDO COM O TIPO DE FERRAMENTA. 
                                                          # Para agente react (https://smith.langchain.com/hub/hwchase17/react)  
                                      )
      
      """ self.agente = create_openai_tools_agent(
                                                llm=llm, # INFORMA A LLM QUE VAI SER USADA PELO AGENTE
                                                tools=self.tools, # PASSANDO PARA A LLM A FERRAMENTA QUE ELA PODE USAR. INSTÂNCIA DA FERRAMENTA
                                                prompt=prompt  
                                                                  # JÁ EXISTEM PROMPTS PRONTOS NO REPOSITÓRIO DO LANGSMITH, DE ACORDO COM O TIPO DE FERRAMENTA. 
                                                                          # Para agente de função (https://smith.langchain.com/hub/hwchase17/openai-functions-agent)
                                                                                
                                             ) """

      print('Prompt:', prompt)

<b>6) Executando o agente com as ferramentas</b>

In [9]:
from langchain.agents import AgentExecutor

agente = AgenteReAct()

executor = AgentExecutor(
                            agent=agente.agente, # O AGENTE QUE VAI SER USADO
                            tools=agente.tools, # FERRAMENTAS QUE O AGENTE PODE OU NÃO USAR
                            verbose=True
                        )

for pergunta in [
                    "Dentre USP e Uni campo, qual universidade você recomenda para a acadêmica Ana ?",
                    "A Universidade Santa Úrsula, no Rio de Janeiro, é uma boa opção para o estudante Marcos ?",
                    "A Cesar School é uma boa universidade para o estudante Marcos ?"                                                                                                                                                                                                                                                          
                ]:
    
    print('\nPergunta: ', pergunta,"\n")
    
    print()
    resposta = executor.invoke({"input": pergunta})
    print(resposta)
    
    

Inicializando ferramenta Dados de Estudante
Inicializando ferramenta Perfil Acadêmico
Inicializando ferramenta Dados Da Universidade
Inicializando ferramenta Dados Da Universidade Web
Prompt: input_variables=['agent_scratchpad', 'input', 'tool_names', 'tools'] input_types={} partial_variables={} metadata={'lc_hub_owner': 'hwchase17', 'lc_hub_repo': 'react', 'lc_hub_commit_hash': 'd15fe3c426f1c4b3f37c9198853e4a86e20c425ca7f4752ec0c9b0e97ca7ea4d'} template='Answer the following questions as best you can. You have access to the following tools:\n\n{tools}\n\nUse the following format:\n\nQuestion: the input question you must answer\nThought: you should always think about what to do\nAction: the action to take, should be one of [{tool_names}]\nAction Input: the input to the action\nObservation: the result of the action\n... (this Thought/Action/Action Input/Observation can repeat N times)\nThought: I now know the final answer\nFinal Answer: the final answer to the original input question\n\